# Modules:

In [1]:
import pandas as pd
import openai
from openai import OpenAI
import time
import json
import os
import sys
from tqdm import tqdm
from dotenv import load_dotenv

# Config:

In [2]:
# --- CONFIGURATION ---
# Load the API key from your specific .env file
load_dotenv("API_keys.env")
API_KEY = os.getenv("OPENAI_API_KEY")
if API_KEY: print("API_KEY loaded succesfully")

# Prompt Technique Toggles
EXAMPLES_ENABLED = False    # 3-shot examples
REASONING_ENABLED = False   # Chain-of-thought reasoning
CONFIDENCE_ENABLED = False  # Confidence score
ROLE_NR = 1                   # Choose the system prompt (role): default = 1
ROLE = {1: "an expert in ESG investment", 
         2: "an expert in classifying posts", 
         3: "a professor of linguistics",
         4: "an academic expert in ESG (Environmental, Social, and Governance) analysis"}[ROLE_NR]

# --- FILES & MODEL ---
MODEL = "GPT5"
MODEL_ID = {"Gem2.5f": "gemini-2.5-flash", "GPT5": "gpt-5"}[MODEL]
OUTPUT_FILE = f"PoC classification_{MODEL}_role{ROLE_NR}{"_examples" if EXAMPLES_ENABLED else ""}{"_reasoning" if REASONING_ENABLED else ""}{"_confidence" if CONFIDENCE_ENABLED else ""}.csv"
INPUT_FILE = "PoC data.csv" if not os.path.exists(OUTPUT_FILE) else OUTPUT_FILE     # Continue with the previous output if output has already been created for this configuration
CATEGORY_FILE = "classification_categories.txt"
EXAMPLES_FILE = f"classification_examples{"_reasoning" if REASONING_ENABLED else ""}{"_confidence" if CONFIDENCE_ENABLED else ""}.txt"

# Script Parameters
TEMPERATURE = 1                       # 0 for strictly deterministic, however, GPT 5 only allows 1 (as it's a reasoning model)
MAX_POSTS = 20                        # "None" for full analysis
POST_START = 0                       # Index of the post where analysis should start (0 is first post)
CHECKPOINT_INTERVAL = 5              # Save after x posts
SLEEP_TIME = 11                        # Break (in seconds) between API calls
RETRIES = 3                             # Amount of retries for 500 / 503 errors

# The 10 official LSEG subcategories
SUBCATEGORIES = [
    "Resource Use", "Emissions", "Innovation", 
    "Workforce", "Human Rights", "Community", "Product Responsibility", 
    "Management", "Shareholders", "CSR Strategy"
]

# Mapping from subcategory to parent category
PARENT_MAP = {
    "Resource Use": "E", "Emissions": "E", "Innovation": "E",
    "Workforce": "S", "Human Rights": "S", "Community": "S", "Product Responsibility": "S",
    "Management": "G", "Shareholders": "G", "CSR Strategy": "G",
    "None": "N"
}

API_KEY loaded succesfully


# Classification algorithm:

In [3]:
client = OpenAI(api_key=API_KEY)

def load_definitions(filepath):
    """Loads ESG definitions from text file. Raises error if file is missing."""
    if not os.path.exists(filepath):
        print(f"CRITICAL ERROR: Definition file '{filepath}' not found.")
        raise FileNotFoundError(f"Missing required file: {filepath}")
    
    with open(filepath, 'r', encoding='utf-8') as f:
        print(filepath, "opened successfully.")
        return f.read()

def load_examples(filepath):
    """Loads few-shot examples from text file. Raises error if file is missing."""
    if not os.path.exists(filepath):
        print(f"CRITICAL ERROR: Examples file '{filepath}' not found.")
        raise FileNotFoundError(f"Missing required file: {filepath}")
        
    with open(filepath, 'r', encoding='utf-8') as f:
        print(filepath, "opened successfully.")
        return f.read()

def build_prompt(post_text, definitions, examples):
    """Dynamically builds the prompt based on configuration."""
    
    # Constructing the dynamic JSON schema description
    schema_fields = ['"Cat_E": "Yes/No"', '"Cat_S": "Yes/No"', '"Cat_G": "Yes/No"', '"Cat_N": "Yes/No"']
    for sub in SUBCATEGORIES:
        clean_sub = sub.replace(" ", "_")
        details = ['"active": "Yes/No"']
        if REASONING_ENABLED: details.append('"reasoning": "string"')
        if CONFIDENCE_ENABLED: details.append('"confidence": float(0-1)')
        schema_fields.append(f'"{clean_sub}": {{{", ".join(details)}}}')

    prompt = f"""
Your task is to analyze LinkedIn posts and classify them into 10 specific LSEG ESG subcategories.

Definitions of the LSEG ESG subcategories:
{definitions}

Instructions:
1. Determine if the article contains explicit thematic evidence of one or more of the 10 LSEG ESG categories. 
You are identifying the presence of a topic, regardless of whether the information is positive, negative, or neutral. 
Only classify based on explicit evidence. NEVER derive or imply categories.
2. If there is absolutely no concrete evidence for any category, assign "N" as the label and "None" as the sublabel. 
Even if you assign "None", you MUST {"provide a justification explaining why the article is not ESG-relevant and" if REASONING_ENABLED else ""} set all scores to 0. 
Never provide an empty output.
{ "3. Provide a detailed reasoning for every active subcategory classification." if REASONING_ENABLED else "" }
{ "4. Provide a confidence score (0.0 to 1.0) for every classification, representing your level of certainty." if CONFIDENCE_ENABLED else "" }

{ f"Examples:\n{examples}" if EXAMPLES_ENABLED else ""}

Strict Output Format (JSON):
{{
    {", ".join(schema_fields)}
}}

Input (LinkedIn Post):
{post_text}
"""
    return prompt

def get_response_schema():
    """Defines the JSON schema in OpenAI's expected format."""
    properties = {
        "Cat_E": {"type": "string", "enum": ["Yes", "No"]},
        "Cat_S": {"type": "string", "enum": ["Yes", "No"]},
        "Cat_G": {"type": "string", "enum": ["Yes", "No"]},
        "Cat_N": {"type": "string", "enum": ["Yes", "No"]},
    }
    
    sub_props = {
        "active": {"type": "string", "enum": ["Yes", "No"]}
    }
    if REASONING_ENABLED:
        sub_props["reasoning"] = {"type": "string"}
    if CONFIDENCE_ENABLED:
        sub_props["confidence"] = {"type": "number"}

    for sub in SUBCATEGORIES:
        clean_sub = sub.replace(" ", "_")
        properties[clean_sub] = {
            "type": "object",
            "properties": sub_props,
            "required": list(sub_props.keys()),
            "additionalProperties": False
        }
    
    return {
        "type": "json_schema",
        "json_schema": {
            "name": "esg_classification",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": properties,
                "required": list(properties.keys()),
                "additionalProperties": False
            }
        }
    }

def classify_post(post_text, definitions, examples, retries):
    """Sends prompt to GPT-5 using Structured Outputs."""
    prompt = build_prompt(post_text, definitions, examples)
    
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": f"You are {ROLE}."},
                    {"role": "user", "content": prompt}
                ],
                temperature=TEMPERATURE,
                response_format=get_response_schema()
            )
            # OpenAI returns the string in .message.content
            return json.loads(response.choices[0].message.content)
            
        except Exception as e:
            if "503" in str(e) or "500" in str(e) or "rate_limit" in str(e).lower():
                print(f"\n[Attempt {attempt + 1}] API Error. Retrying...")
                time.sleep(10 * (attempt + 1))
                continue
            return {"error": str(e)}
            
    return {"error": "Maximum retries reached."}

def main():
    # 1. Verification of required files
    try:
        definitions = load_definitions(CATEGORY_FILE)
        examples = load_examples(EXAMPLES_FILE)
    except FileNotFoundError:
        sys.exit(1)

    if not os.path.exists(INPUT_FILE):
        print(f"Error: Input file '{INPUT_FILE}' not found.")
        return

    # 2. Load Data
    df = pd.read_csv(INPUT_FILE)
    print(INPUT_FILE, "opened succesfully")
    # Expected columns: Company, Date, Link, Text
    text_col = "Text" 

    # 3. Initialize Columns
    # Ensure main category columns exist
    for cat in ["Cat_E", "Cat_S", "Cat_G", "Cat_N"]:
        if cat not in df.columns: df[cat] = None
        
    # Ensure subcategory columns exist
    for sub in SUBCATEGORIES:
        clean_sub = sub.replace(" ", "_")
        cols = [f"{clean_sub}_active"]
        if REASONING_ENABLED: cols.append(f"{clean_sub}_reasoning")
        if CONFIDENCE_ENABLED: cols.append(f"{clean_sub}_confidence")
        
        for col in cols:
            if col not in df.columns: df[col] = None

    # Calculate range
    end_index = len(df)
    if MAX_POSTS is not None:
        end_index = min(POST_START + MAX_POSTS, len(df))

    print(f"Processing posts from index {POST_START} to {end_index}...")

    # 4. Processing Loop
    for index in tqdm(range(POST_START, end_index), desc="Classifying"):
        # Skip if already successfully processed
        if pd.notna(df.at[index, 'Cat_E']) and "ERROR" not in str(df.at[index, 'Cat_N']):
            continue

        post_content = df.at[index, text_col]
        result = classify_post(post_content, definitions, examples, RETRIES)

        if "error" in result:
            df.at[index, 'Cat_N'] = "ERROR: " + result["error"]
        else:
            # Map Main Categories
            for cat in ["Cat_E", "Cat_S", "Cat_G", "Cat_N"]:
                df.at[index, cat] = result.get(cat, "No")

            # Map Subcategories directly
            for sub in SUBCATEGORIES:
                clean_sub = sub.replace(" ", "_")
                sub_data = result.get(clean_sub, {})
                
                df.at[index, f"{clean_sub}_active"] = sub_data.get("active", "No")
                if REASONING_ENABLED:
                    df.at[index, f"{clean_sub}_reasoning"] = sub_data.get("reasoning", "")
                if CONFIDENCE_ENABLED:
                    df.at[index, f"{clean_sub}_confidence"] = sub_data.get("confidence", 0.0)

        # Checkpoint
        if (index + 1) % CHECKPOINT_INTERVAL == 0:
            df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
        
        time.sleep(SLEEP_TIME)

    # Final Save
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"\nProcessing complete! Results saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

classification_categories.txt opened successfully.
classification_examples.txt opened successfully.
PoC data.csv opened succesfully
Processing posts from index 0 to 20...


Classifying: 100%|██████████| 20/20 [07:08<00:00, 21.44s/it]


Processing complete! Results saved to PoC classification_GPT5_role1.csv
